# Génération de la couche Bronze

Ce notebook transforme les trois sources du dossier `datas` en fichiers CSV UTF-8 dans `artifacts/ingestion/bronze`. Les deux fichiers CSV produisent chacun une sortie. Le dump SQL produit une sortie par table, soit quatre fichiers Bronze au total.

## Règles appliquées

- les en-têtes et l’ordre des colonnes d’origine sont conservés ;
- chaque valeur non sensible est copiée sous forme de texte, sans changement de casse, d’espaces ou de contenu ;
- `operator_name` et `operator_badge` sont remplacés par un SHA-256 salé stable ;
- chaque colonne d’origine est immédiatement suivie de `<colonne>_type` ;
- la dernière colonne, `row_sha256`, contient le SHA-256 salé des seules valeurs d’origine ;
- les noms de sortie sont suffixés par `AAAAMMJJ_HHMMSS` ;
- le traitement s’arrête à la première erreur avec un diagnostic explicite ;
- un manifeste JSON horodaté consigne les dimensions, contrôles, ratios de doublons, indices de confiance et erreurs du lot.

## 1. Imports et paramètres

pandas lit et écrit les fichiers plats. SQLAlchemy charge le dump dans une base SQLite en mémoire et expose ses tables. Le sel n’est jamais stocké dans le notebook : il provient de la variable utilisateur Windows `INDUSENSE_HASH_SALT`.

In [1]:
from __future__ import annotations

import csv
import hashlib
import json
import os
import re
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any

import pandas as pd
from sqlalchemy import MetaData, String, cast, create_engine, inspect, select
from sqlalchemy.sql.sqltypes import Boolean, Date, DateTime, Float, Integer, Numeric

INPUT_DIRECTORY = Path("datas")
OUTPUT_DIRECTORY = Path("artifacts/ingestion/bronze")
IDENTITY_OUTPUT_DIRECTORY = Path("artifacts/ingestion/identities")
HASH_COLUMN = "row_sha256"
HASH_SEPARATOR = "\x1f"
SALT_SEPARATOR = "\x1e"
SENSITIVE_COLUMNS = {"operator_name", "operator_badge"}


@dataclass(frozen=True)
class BronzeResult:
    source: Path
    dataset: str
    output: Path
    rows: int
    source_columns: int
    output_columns: int

## 2. Fonctions communes

La transformation travaille uniquement avec des chaînes. Pour les CSV, les valeurs sont lues directement comme texte et les valeurs vides ne sont pas converties en `NaN`. Pour SQL, chaque valeur sélectionnée est convertie en texte par le moteur.

Le hash est calculé sur les valeurs d’origine dans l’ordre des colonnes. Aucune normalisation de contenu n’est appliquée. Le caractère de contrôle `\x1F` sert uniquement de séparateur entre les valeurs afin de réduire les ambiguïtés de concaténation. Le sel est préfixé et séparé par `\x1E`.

In [2]:
def load_hash_salt() -> str:
    salt = os.environ.get("INDUSENSE_HASH_SALT")
    if not salt and os.name == "nt":
        import winreg
        try:
            with winreg.OpenKey(winreg.HKEY_CURRENT_USER, "Environment") as key:
                salt, _ = winreg.QueryValueEx(key, "INDUSENSE_HASH_SALT")
        except FileNotFoundError:
            salt = None
    if not salt:
        raise EnvironmentError(
            "La variable INDUSENSE_HASH_SALT est absente ou vide."
        )
    return str(salt)


def detect_charset(path: Path) -> tuple[str, str]:
    content = path.read_bytes()
    if content.startswith(b"\xef\xbb\xbf"):
        return "UTF-8 BOM", "utf-8-sig"
    for label, codec in (
        ("UTF-8", "utf-8"),
        ("Windows-1252", "cp1252"),
        ("ISO-8859-1", "iso-8859-1"),
    ):
        try:
            content.decode(codec)
            return label, codec
        except UnicodeDecodeError:
            continue
    raise UnicodeError(f"Impossible de déterminer le charset de {path}")


def detect_delimiter(path: Path, encoding: str) -> str:
    with path.open("r", encoding=encoding, newline="") as stream:
        sample = stream.read(16_384)
    if not sample:
        raise ValueError("Le fichier CSV est vide.")
    try:
        return csv.Sniffer().sniff(sample, delimiters=",;\t|").delimiter
    except csv.Error as exc:
        raise ValueError("Séparateur CSV indéterminable.") from exc


def infer_text_type(value: str, *, is_null: bool = False) -> str:
    if is_null:
        return "NULL"
    if value == "":
        return "EMPTY"
    if re.fullmatch(r"(?i:true|false)", value):
        return "BOOLEAN"
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}[ T]\d{2}:\d{2}:\d{2}(?:\.\d+)?(?:Z|[+-]\d{2}(?::?\d{2})?)?", value):
        return "DATETIME"
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", value):
        return "DATE"
    if re.fullmatch(r"\d{2}:\d{2}(?::\d{2}(?:\.\d+)?)?", value):
        return "TIME"
    if re.fullmatch(r"[+-]?\d+", value):
        return "INTEGER"
    if re.fullmatch(r"[+-]?(?:\d+\.\d*|\d*\.\d+)(?:[eE][+-]?\d+)?|[+-]?\d+[eE][+-]?\d+", value):
        return "FLOAT"
    return "TEXT"


def compute_row_hash(values: list[str], salt: str) -> str:
    payload = salt + SALT_SEPARATOR + HASH_SEPARATOR.join(values)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def hash_sensitive_value(value: str, column: str, salt: str) -> str:
    payload = salt + SALT_SEPARATOR + column + HASH_SEPARATOR + value
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def anonymize_sensitive_columns(frame: pd.DataFrame, salt: str) -> pd.DataFrame:
    anonymized = frame.copy()
    for column in SENSITIVE_COLUMNS.intersection(anonymized.columns):
        anonymized[column] = anonymized[column].map(
            lambda value: "" if value == "" else hash_sensitive_value(str(value), column, salt)
        )
    return anonymized


def build_bronze_frame(
    source_frame: pd.DataFrame,
    salt: str,
    explicit_types: dict[str, str] | None = None,
    null_masks: dict[str, pd.Series] | None = None,
) -> pd.DataFrame:
    original_columns = [str(column) for column in source_frame.columns]
    text_frame = source_frame.copy()
    text_frame.columns = original_columns
    text_frame = text_frame.astype("string").fillna("")

    output = pd.DataFrame(index=text_frame.index)
    for column in original_columns:
        values = text_frame[column].astype(str)
        output[column] = values
        if explicit_types and column in explicit_types:
            inferred = pd.Series(explicit_types[column], index=text_frame.index, dtype="string")
            if null_masks and column in null_masks:
                inferred = inferred.mask(null_masks[column], "NULL")
            output[f"{column}_type"] = inferred
        else:
            output[f"{column}_type"] = values.map(infer_text_type)

    output[HASH_COLUMN] = [
        compute_row_hash([str(value) for value in row], salt)
        for row in text_frame[original_columns].itertuples(index=False, name=None)
    ]
    return output


def write_bronze_csv(frame: pd.DataFrame, output: Path) -> None:
    output.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(
        output,
        index=False,
        encoding="utf-8",
        lineterminator="\n",
        quoting=csv.QUOTE_ALL,
    )

## 3. Traitement des fichiers plats

`dtype=str`, `keep_default_na=False` et `na_filter=False` empêchent pandas de convertir les valeurs. Le fichier Bronze conserve ainsi le texte logique de chaque champ CSV. Toutes les cellules sont entourées de guillemets lors de l’écriture pour expliciter leur nature textuelle.

In [3]:
def process_flat_file(
    source: Path, output_directory: Path, timestamp: str, salt: str
) -> BronzeResult:
    _, encoding = detect_charset(source)
    delimiter = detect_delimiter(source, encoding)
    source_frame = pd.read_csv(
        source,
        encoding=encoding,
        sep=delimiter,
        dtype=str,
        keep_default_na=False,
        na_filter=False,
    )
    bronze_frame = build_bronze_frame(anonymize_sensitive_columns(source_frame, salt), salt)
    output = output_directory / f"{timestamp}__{source.stem}.csv"
    write_bronze_csv(bronze_frame, output)
    return BronzeResult(
        source=source.resolve(),
        dataset=source.stem,
        output=output.resolve(),
        rows=len(source_frame),
        source_columns=len(source_frame.columns),
        output_columns=len(bronze_frame.columns),
    )

## 4. Traitement du dump SQL

Le dump PostgreSQL fourni contient deux tables. Deux adaptations syntaxiques ciblées permettent son chargement dans SQLite en mémoire, sans modifier la source. SQLAlchemy réfléchit ensuite les tables et sélectionne chaque valeur avec un `CAST` texte. Les types associés proviennent du schéma SQL réfléchi ; une valeur SQL `NULL` reçoit toujours le type `NULL`.

In [4]:
def sqlite_compatible_sql(sql: str) -> str:
    sql = re.sub(r"\bTIMESTAMPTZ\b", "TIMESTAMP", sql, flags=re.IGNORECASE)
    return re.sub(r"\bNOW\s*\(\s*\)", "CURRENT_TIMESTAMP", sql, flags=re.IGNORECASE)


def declared_sql_type(column: Any) -> str:
    if isinstance(column.type, Boolean):
        return "BOOLEAN"
    if isinstance(column.type, DateTime):
        return "DATETIME"
    if isinstance(column.type, Date):
        return "DATE"
    if isinstance(column.type, Integer):
        return "INTEGER"
    if isinstance(column.type, (Numeric, Float)):
        return "FLOAT"
    return "TEXT"


def load_sql_tables(source: Path) -> tuple[Any, MetaData]:
    _, encoding = detect_charset(source)
    sql = source.read_text(encoding=encoding)
    if not sql.strip():
        raise ValueError("Le fichier SQL est vide.")
    engine = create_engine("sqlite+pysqlite:///:memory:")
    try:
        raw_connection = engine.raw_connection()
        try:
            raw_connection.executescript(sqlite_compatible_sql(sql))
            raw_connection.commit()
        finally:
            raw_connection.close()
        metadata = MetaData()
        metadata.reflect(bind=engine)
        if not metadata.tables:
            raise ValueError("Aucune table trouvée dans le fichier SQL.")
        return engine, metadata
    except Exception:
        engine.dispose()
        raise


def process_sql_file(
    source: Path, output_directory: Path, timestamp: str, salt: str
) -> list[BronzeResult]:
    engine, metadata = load_sql_tables(source)
    results: list[BronzeResult] = []
    try:
        with engine.connect() as connection:
            for table_name in inspect(engine).get_table_names():
                table = metadata.tables[table_name]
                columns = list(table.columns)
                statement = select(*[cast(column, String).label(column.name) for column in columns])
                rows = connection.execute(statement).mappings().all()
                source_frame = pd.DataFrame(
                    [[row[column.name] for column in columns] for row in rows],
                    columns=[column.name for column in columns],
                )
                null_masks = {
                    column.name: source_frame[column.name].isna() for column in columns
                }
                explicit_types = {
                    column.name: declared_sql_type(column) for column in columns
                }
                bronze_frame = build_bronze_frame(
                    source_frame, salt, explicit_types=explicit_types, null_masks=null_masks
                )
                output = output_directory / f"{timestamp}__{table_name}.csv"
                write_bronze_csv(bronze_frame, output)
                persisted = pd.read_csv(
                    output, dtype=str, keep_default_na=False, na_filter=False
                )
                pd.testing.assert_frame_equal(
                    source_frame.astype("string").fillna("").reset_index(drop=True),
                    persisted[[column.name for column in columns]].reset_index(drop=True),
                    check_dtype=False,
                )
                results.append(BronzeResult(
                    source=source.resolve(),
                    dataset=table_name,
                    output=output.resolve(),
                    rows=len(source_frame),
                    source_columns=len(columns),
                    output_columns=len(bronze_frame.columns),
                ))
    finally:
        engine.dispose()
    return results

## 5. Orchestration et arrêt sur erreur

Un horodatage unique est créé au début du lot afin que les quatre fichiers, les quatre cartes d’identité et leur rapport JSON soient identifiables comme une même génération. Tous les noms commencent par `batchID__`.

Le rapport mesure le ratio de lignes en doublon exact et attribue un indice de confiance à chaque ligne atypique. Une anomalie correspond à une valeur numérique à plus de trois écarts-types ou à une valeur catégorielle présente dans moins de 1 % du lot. Les champs personnels sont exclus de ce calcul. Le traitement reste séquentiel et s’interrompt dès qu’une source échoue. Le manifeste est écrit atomiquement, y compris en cas d’échec, avec les fichiers déjà produits, les contrôles réussis et le descriptif de l’erreur.

In [5]:
def _markdown_value(value: Any) -> str:
    if value is None or pd.isna(value):
        return "N/A"
    if isinstance(value, float):
        return f"{value:.10g}"
    return str(value).replace("|", r"\|").replace("\n", " ")


def write_identity_card(
    source: Path, dataset: str, frame: pd.DataFrame, batch_id: str,
    output_directory: Path, table: str | None = None,
) -> Path:
    column_rows: list[tuple[str, str, Any, Any, Any, Any]] = []
    for column in frame.columns:
        values = frame[column].astype("string")
        numeric = pd.to_numeric(values, errors="coerce")
        numeric_ratio = numeric.notna().mean() if len(values) else 0
        dates = pd.to_datetime(values, errors="coerce", format="mixed")
        date_ratio = dates.notna().mean() if len(values) else 0
        if numeric_ratio >= 0.9:
            clean = numeric.dropna()
            kind = "NUMERIC"
            column_rows.append((str(column), kind, clean.min(), clean.max(), clean.mean(), clean.median()))
        elif date_ratio >= 0.9:
            clean = dates.dropna()
            kind = "DATETIME" if values.str.contains(r"[ T]\d{2}:\d{2}", regex=True).any() else "DATE"
            column_rows.append((str(column), kind, clean.min(), clean.max(), None, None))
        elif str(column).casefold() == "id" or str(column).casefold().endswith("_id"):
            column_rows.append((str(column), "IDENTITY", None, None, None, None))
        else:
            column_rows.append((str(column), "TEXT", None, None, None, None))

    lines = [
        f"* BatchID: {batch_id}",
        f"* Source: {source.resolve()}",
        f"* Dataset: {dataset}",
        f"* Filsize: {source.stat().st_size} o",
        f"* Rows Number: {len(frame)}",
        f"* Cols Number: {len(frame.columns)}",
        "", "## Column list", "",
        "| Colname | Infered Type |", "| --- | --- |",
    ]
    lines.extend(f"| {_markdown_value(name)} | {kind} |" for name, kind, *_ in column_rows)
    lines.extend([
        "", "## Numeric and temporal statistics", "",
        "| Col name | Infered Type | Min value | Max value | Mean value | Median value |",
        "| --- | --- | ---: | ---: | ---: | ---: |",
    ])
    for name, kind, minimum, maximum, mean, median in column_rows:
        if kind in {"NUMERIC", "DATE", "DATETIME"}:
            lines.append("| " + " | ".join(_markdown_value(value) for value in (name, kind, minimum, maximum, mean, median)) + " |")
    output_directory.mkdir(parents=True, exist_ok=True)
    prefix = f"{batch_id}__identity_{table or dataset}"
    card = output_directory / f"{prefix}.md"
    card.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return card.resolve()


def generate_identity_cards(source: Path, batch_id: str) -> list[Path]:
    if source.suffix.casefold() == ".csv":
        _, encoding = detect_charset(source)
        frame = pd.read_csv(source, encoding=encoding, sep=detect_delimiter(source, encoding), dtype=str, keep_default_na=False, na_filter=False)
        return [write_identity_card(source, source.stem, frame, batch_id, IDENTITY_OUTPUT_DIRECTORY)]
    engine, metadata = load_sql_tables(source)
    cards: list[Path] = []
    try:
        with engine.connect() as connection:
            for table_name in inspect(engine).get_table_names():
                table = metadata.tables[table_name]
                rows = connection.execute(select(*[cast(column, String).label(column.name) for column in table.columns])).mappings().all()
                frame = pd.DataFrame(rows, columns=[column.name for column in table.columns])
                cards.append(write_identity_card(source, source.stem, frame, batch_id, IDENTITY_OUTPUT_DIRECTORY, table_name))
    finally:
        engine.dispose()
    return cards


def analyze_quality(output: Path, dataset: str) -> dict[str, Any]:
    bronze = pd.read_csv(output, dtype=str, keep_default_na=False, na_filter=False)
    columns = list(bronze.columns[0:-1:2])
    frame = bronze[columns]
    duplicate_mask = frame.duplicated(keep=False)
    reasons: dict[int, list[str]] = {index: [] for index in frame.index}
    eligible_columns = 0
    for column in columns:
        if column in SENSITIVE_COLUMNS:
            continue
        numeric = pd.to_numeric(frame[column], errors="coerce")
        if numeric.notna().mean() >= 0.9 and numeric.std(ddof=0) > 0:
            eligible_columns += 1
            z_score = ((numeric - numeric.mean()) / numeric.std(ddof=0)).abs()
            for index in z_score[z_score > 3].index:
                reasons[index].append(f"{column}: z-score > 3")
            continue
        unique_count = frame[column].nunique(dropna=False)
        if 1 < unique_count <= min(50, max(2, int(len(frame) * 0.2))):
            eligible_columns += 1
            frequencies = frame[column].value_counts(dropna=False, normalize=True)
            rare_values = set(frequencies[frequencies < 0.01].index)
            for index in frame.index[frame[column].isin(rare_values)]:
                reasons[index].append(f"{column}: valeur rare < 1 %")
    outliers = [
        {"row_number": int(index) + 1, "confidence": round(1 - len(row_reasons) / eligible_columns, 4), "reasons": row_reasons}
        for index, row_reasons in reasons.items() if row_reasons
    ] if eligible_columns else []
    return {
        "duplicate_ratio": round(float(duplicate_mask.mean()), 6) if len(frame) else 0.0,
        "duplicate_rows": int(duplicate_mask.sum()),
        "confidence": {
            "method": "numeric z-score > 3; categorical frequency < 1 %; sensitive fields excluded",
            "eligible_columns": eligible_columns,
            "outlier_rows": outliers,
            "outlier_ratio": round(len(outliers) / len(frame), 6) if len(frame) else 0.0,
        },
    }


def run_batch_checks(results: list[BronzeResult], salt: str) -> list[dict[str, Any]]:
    checks: list[dict[str, Any]] = []
    checks.append({
        "name": "generated_file_count", "status": "PASS",
        "expected": 4, "actual": len(results),
    })
    if len(results) != 4:
        raise AssertionError(f"4 fichiers attendus, {len(results)} générés.")

    for result in results:
        result.output.read_text(encoding="utf-8")
        checks.append({"name": "utf8_encoding", "dataset": result.dataset, "status": "PASS"})
        bronze = pd.read_csv(
            result.output, dtype=str, keep_default_na=False, na_filter=False
        )
        original_columns = list(bronze.columns[0:-1:2])
        expected_headers = [
            header for column in original_columns
            for header in (column, f"{column}_type")
        ] + [HASH_COLUMN]
        if list(bronze.columns) != expected_headers:
            raise AssertionError(f"Entêtes Bronze invalides pour {result.dataset}.")
        checks.append({"name": "interleaved_type_columns", "dataset": result.dataset, "status": "PASS"})

        if not bronze[HASH_COLUMN].str.fullmatch(r"[0-9a-f]{64}").all():
            raise AssertionError(f"Format SHA-256 invalide pour {result.dataset}.")
        recomputed = [
            compute_row_hash([str(value) for value in row], salt)
            for row in bronze[original_columns].itertuples(index=False, name=None)
        ]
        if recomputed != bronze[HASH_COLUMN].tolist():
            raise AssertionError(f"Empreinte invalide pour {result.dataset}.")
        checks.append({"name": "sha256_recalculation", "dataset": result.dataset, "status": "PASS"})
        sensitive_columns = SENSITIVE_COLUMNS.intersection(original_columns)
        if sensitive_columns:
            if not all(bronze[column].str.fullmatch(r"[0-9a-f]{64}").all() for column in sensitive_columns):
                raise AssertionError(f"Protection des données sensibles invalide pour {result.dataset}.")
            checks.append({"name": "sensitive_columns_hashed", "dataset": result.dataset, "status": "PASS"})
    return checks


def write_operation_report(
    output_directory: Path, batch_id: str, started_at: datetime,
    status: str, results: list[BronzeResult], checks: list[dict[str, Any]],
    errors: list[dict[str, str]], identity_cards: list[Path],
) -> Path:
    ended_at = datetime.now().astimezone()
    payload = {
        "batch_id": batch_id,
        "status": status,
        "started_at": started_at.isoformat(),
        "ended_at": ended_at.isoformat(),
        "duration_seconds": round((ended_at - started_at).total_seconds(), 3),
        "generated_files": len(results),
        "datasets": [
            {
                "source": str(item.source), "name": item.dataset,
                "rows": item.rows, "source_columns": item.source_columns,
                "bronze_columns": item.output_columns,
                "output": str(item.output),
                "size_bytes": item.output.stat().st_size,
                "quality": analyze_quality(item.output, item.dataset),
            }
            for item in results
        ],
        "identity_cards": [str(card) for card in identity_cards],
        "checks": checks,
        "errors": errors,
        "hash": {"algorithm": "SHA-256", "salt_source": "INDUSENSE_HASH_SALT"},
    }
    output_directory.mkdir(parents=True, exist_ok=True)
    report = output_directory / f"{batch_id}__bronze_report.json"
    temporary_report = output_directory / f".{batch_id}__bronze_report.json.tmp"
    temporary_report.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )
    temporary_report.replace(report)
    return report.resolve()


def generate_bronze_files(
    input_directory: Path = INPUT_DIRECTORY,
    output_directory: Path = OUTPUT_DIRECTORY,
) -> tuple[list[BronzeResult], Path]:
    started_at = datetime.now().astimezone()
    timestamp = started_at.strftime("%Y%m%d_%H%M%S")
    salt = load_hash_salt()
    flat_sources = sorted(input_directory.glob("*.csv"))
    sql_sources = sorted(input_directory.glob("*.sql.txt"))
    if not flat_sources and not sql_sources:
        raise FileNotFoundError(f"Aucune source trouvée dans {input_directory.resolve()}")

    results: list[BronzeResult] = []
    checks: list[dict[str, Any]] = []
    identity_cards: list[Path] = []
    for source in [*flat_sources, *sql_sources]:
        try:
            identity_cards.extend(generate_identity_cards(source, timestamp))
            if source.suffix.casefold() == ".csv":
                results.append(process_flat_file(source, output_directory, timestamp, salt))
            else:
                results.extend(process_sql_file(source, output_directory, timestamp, salt))
        except Exception as exc:
            errors = [{
                "source": str(source.resolve()),
                "type": type(exc).__name__,
                "message": str(exc),
            }]
            report_path = write_operation_report(
                output_directory, timestamp, started_at, "FAILED",
                results, checks, errors, identity_cards,
            )
            print(f"ERREUR pendant le traitement de {source.resolve()}")
            print(f"{type(exc).__name__}: {exc}")
            print(f"Rapport d’échec : {report_path}")
            raise RuntimeError(f"Traitement Bronze interrompu sur {source.name}") from exc

    try:
        checks = run_batch_checks(results, salt)
    except Exception as exc:
        errors = [{
            "source": "batch_validation",
            "type": type(exc).__name__,
            "message": str(exc),
        }]
        report_path = write_operation_report(
            output_directory, timestamp, started_at, "FAILED",
            results, checks, errors, identity_cards,
        )
        print(f"ERREUR pendant les contrôles du lot : {type(exc).__name__}: {exc}")
        print(f"Rapport d’échec : {report_path}")
        raise RuntimeError("Contrôles Bronze en échec") from exc

    report_path = write_operation_report(
        output_directory, timestamp, started_at, "SUCCESS", results, checks, [], identity_cards
    )

    print(f"Traitement Bronze terminé : {len(results)} fichier(s) généré(s).")
    for result in results:
        print(
            f"- {result.dataset}: {result.rows} ligne(s), "
            f"{result.source_columns} colonne(s) source -> {result.output}"
        )
    print(f"Rapport d’opération -> {report_path}")
    return results, report_path

## 6. Génération des fichiers Bronze

In [6]:
results, report_path = generate_bronze_files()

summary = pd.DataFrame([
    {
        "Source": item.source.name,
        "Jeu de données": item.dataset,
        "Lignes": item.rows,
        "Colonnes source": item.source_columns,
        "Colonnes Bronze": item.output_columns,
        "Fichier Bronze": str(item.output),
    }
    for item in results
])
print(f"Rapport : {report_path}")
summary

Traitement Bronze terminé : 4 fichier(s) généré(s).
- releves_incidents: 1245 ligne(s), 18 colonne(s) source -> D:\indusense\artifacts\ingestion\bronze\20261127_102852__releves_incidents.csv
- telemetry: 135626 ligne(s), 7 colonne(s) source -> D:\indusense\artifacts\ingestion\bronze\20261127_102852__telemetry.csv
- machine: 15 ligne(s), 11 colonne(s) source -> D:\indusense\artifacts\ingestion\bronze\20261127_102852__machine.csv
- maintenance: 115 ligne(s), 11 colonne(s) source -> D:\indusense\artifacts\ingestion\bronze\20261127_102852__maintenance.csv
Rapport d’opération -> D:\indusense\artifacts\ingestion\bronze\20261127_102852__bronze_report.json
Rapport : D:\indusense\artifacts\ingestion\bronze\20261127_102852__bronze_report.json


,Source,Jeu de données,Lignes,Colonnes source,Colonnes Bronze,Fichier Bronze
0,releves_incidents.csv,releves_incidents,1245,18,37,D:\indusense\artifacts\ingestion\bronze\202611...
1,telemetry.csv,telemetry,135626,7,15,D:\indusense\artifacts\ingestion\bronze\202611...
2,machine.sql.txt,machine,15,11,23,D:\indusense\artifacts\ingestion\bronze\202611...
3,machine.sql.txt,maintenance,115,11,23,D:\indusense\artifacts\ingestion\bronze\202611...


## 7. Contrôles

Les contrôles vérifient le nombre de sorties, les dimensions, l’alternance des colonnes, l’encodage UTF-8, la conservation des valeurs non sensibles, le hash des champs personnels, le recalcul des empreintes et le manifeste. Le sel reste secret et n’est jamais affiché.

In [7]:
expected = {
    "releves_incidents": (1245, 18),
    "telemetry": (135626, 7),
    "machine": (15, 11),
    "maintenance": (115, 11),
}
assert len(results) == 4
salt = load_hash_salt()

for result in results:
    expected_rows, expected_source_columns = expected[result.dataset]
    assert result.rows == expected_rows
    assert result.source_columns == expected_source_columns
    assert result.output_columns == expected_source_columns * 2 + 1
    result.output.read_text(encoding="utf-8")

    bronze = pd.read_csv(
        result.output, dtype=str, keep_default_na=False, na_filter=False
    )
    original_columns = list(bronze.columns[0:-1:2])
    expected_headers = [
        header
        for column in original_columns
        for header in (column, f"{column}_type")
    ] + [HASH_COLUMN]
    assert list(bronze.columns) == expected_headers
    assert bronze[HASH_COLUMN].str.fullmatch(r"[0-9a-f]{64}").all()
    recomputed = [
        compute_row_hash([str(value) for value in row], salt)
        for row in bronze[original_columns].itertuples(index=False, name=None)
    ]
    assert recomputed == bronze[HASH_COLUMN].tolist()

for result in (item for item in results if item.source.suffix.casefold() == ".csv"):
    _, encoding = detect_charset(result.source)
    delimiter = detect_delimiter(result.source, encoding)
    source_frame = pd.read_csv(
        result.source, encoding=encoding, sep=delimiter, dtype=str,
        keep_default_na=False, na_filter=False,
    )
    bronze = pd.read_csv(
        result.output, dtype=str, keep_default_na=False, na_filter=False
    )
    non_sensitive_columns = [column for column in source_frame.columns if column not in SENSITIVE_COLUMNS]
    pd.testing.assert_frame_equal(
        source_frame[non_sensitive_columns].reset_index(drop=True),
        bronze[non_sensitive_columns].reset_index(drop=True),
        check_dtype=False,
    )
    for column in SENSITIVE_COLUMNS.intersection(source_frame.columns):
        assert bronze[column].str.fullmatch(r"[0-9a-f]{64}").all()
        assert not bronze[column].isin(source_frame[column]).any()

report_text = report_path.read_text(encoding="utf-8")
report = json.loads(report_text)
assert report["status"] == "SUCCESS"
assert report["generated_files"] == 4
assert len(report["datasets"]) == 4
assert report["errors"] == []
assert report["checks"]
assert all(check["status"] == "PASS" for check in report["checks"])
assert report["hash"] == {
    "algorithm": "SHA-256", "salt_source": "INDUSENSE_HASH_SALT"
}
assert salt not in report_text
assert len(report["identity_cards"]) == 4
assert all(Path(card).is_file() for card in report["identity_cards"])
assert all(Path(item["output"]).name.startswith(report["batch_id"] + "__") for item in report["datasets"])
assert Path(report_path).name.startswith(report["batch_id"] + "__")
assert all(Path(card).name.startswith(report["batch_id"] + "__") for card in report["identity_cards"])
assert all("duplicate_ratio" in item["quality"] for item in report["datasets"])

print("Contrôles réussis : 4 sorties UTF-8, données sensibles hashées, cartes, qualité, hash et rapport validés.")

Contrôles réussis : 4 sorties UTF-8, données sensibles hashées, cartes, qualité, hash et rapport validés.


## Synthèse des choix

- **CSV UTF-8 avec guillemets systématiques** : les sorties restent des fichiers plats et toutes les valeurs sont explicitement textuelles.
- **Une sortie par source logique** : deux fichiers plats et deux tables SQL produisent quatre fichiers.
- **Colonnes intercalées** : chaque valeur est immédiatement suivie de son type, ce qui facilite les contrôles ligne par ligne.
- **Aucune normalisation des valeurs** : le hash reprend le texte sans `strip`, changement de casse ou remplacement.
- **Sel externe persistant** : la variable utilisateur Windows protège le sel et stabilise les empreintes entre les exécutions.
- **Horodatage commun au lot** : les quatre noms de fichiers et le rapport portent la date et l’heure d’une même génération.
- **Rapport atomique** : un manifeste JSON `SUCCESS` ou `FAILED` conserve les dimensions, contrôles et erreurs sans exposer le sel.